In [ ]:
#1、完成对facebook数据的k近邻的机器学习，并通过网格搜索找到最佳的kneighbors
#预处理
import pandas as pd

data = pd.read_csv("FBlocation/train.csv")

data.dropna(inplace=True)
print(data.head)

data = data.query("x>1.0 & x < 1.25 & y>2.5 & y < 2.75")
print(data.shape)

<bound method NDFrame.head of             row_id       x       y  accuracy    time    place_id
0                0  0.7941  9.0809        54  470702  8523065625
1                1  5.9567  4.7968        13  186555  1757726713
2                2  8.3078  7.0407        74  322648  1137537235
3                3  7.3665  2.5165        65  704587  6567393236
4                4  4.0961  1.1307        31  472130  7440663949
...            ...     ...     ...       ...     ...         ...
29118016  29118016  6.5133  1.1435        67  399740  8671361106
29118017  29118017  5.9186  4.4134        67  125480  9077887898
29118018  29118018  2.9993  6.3680        67  737758  2838334300
29118019  29118019  4.0637  8.0061        70  764975  1007355847
29118020  29118020  7.4523  2.0871        17  102842  7028698129

[29118021 rows x 6 columns]>
(17710, 6)


In [ ]:
#数据特征与标签处理
y = data['place_id']
X = data.drop(['place_id', 'row_id'], axis=1)

X['time'] = pd.to_datetime(X['time'], unit='s')
X['hour'] = X['time'].dt.hour
X['day'] = X['time'].dt.day
X['weekday'] = X['time'].dt.weekday
print(X)
X = X.drop('time', axis=1)

print(X.shape)  # 打印特征矩阵的形状
print(y.shape)  # 打印标签向量的形状
print(X.columns.tolist())  # 打印所有特征列名
print(X.head())  # 打印特征矩阵的前5行

unique_places = y.nunique()  # 计算标签中不同地点的数量
print(unique_places)  # 打印不同地点的总数

place_counts = y.value_counts()  # 统计每个地点出现的次数
print(place_counts)  # 打印每个地点的出现次数

print(place_counts.max())  # 打印出现次数最多的地点的频次
print(place_counts.min())  # 打印出现次数最少的地点的频次
print(place_counts.mean())  # 打印每个地点出现次数的平均值


               x       y  accuracy                time  hour  day  weekday
600       1.2214  2.7023        17 1970-01-01 18:09:40    18    1        3
957       1.1832  2.6891        58 1970-01-10 02:11:10     2   10        5
4345      1.1935  2.6550        11 1970-01-05 15:08:02    15    5        0
4735      1.1452  2.6074        49 1970-01-06 23:03:03    23    6        1
5580      1.0089  2.7287        19 1970-01-09 11:26:50    11    9        4
...          ...     ...       ...                 ...   ...  ...      ...
29100203  1.0129  2.6775        12 1970-01-01 10:33:56    10    1        3
29108443  1.1474  2.6840        36 1970-01-07 23:22:04    23    7        2
29109993  1.0240  2.7238        62 1970-01-08 15:03:14    15    8        3
29111539  1.2032  2.6796        87 1970-01-04 00:53:41     0    4        6
29112154  1.1070  2.5419       178 1970-01-08 23:01:07    23    8        3

[17710 rows x 7 columns]
(17710, 6)
(17710,)
['x', 'y', 'accuracy', 'hour', 'day', 'weekday']
     

In [ ]:
#样本过滤

places_to_keep = place_counts[place_counts > 2].index

mask = y.isin(places_to_keep)
X_filtered = X[mask]
y_filtered = y[mask]

print(len(X))               # 打印原始特征数据X的样本数量
print(len(X_filtered))      # 打印过滤后特征数据X_filtered的样本数量
print(len(y.unique()))      # 打印原始标签y中唯一类别的数量
print(len(y_filtered.unique()))  # 打印过滤后标签y_filtered中唯一类别的数量

X = X_filtered
y = y_filtered

print(X.shape)
print(y.shape)

17710
17086
805
295
(17086, 6)
(17086,)


In [ ]:
#数据集划分及KNN评估
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X_train))  # 输出训练集样本数量
print(len(X_test))    # 输出测试集样本数量
print(y_train.unique())  # 输出训练集中标签的唯一值
print(y_test.unique())   # 输出测试集中标签的唯一值

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

accuracy = knn.score(X_test, y_test)
print(f"KNN模型准确率：{accuracy:.4f}")


13668
3418
[8258328058 1097200869 7536975002 8606102186 2946102544 2700174925
 8048985799 6603539415 3322359335 4000153867 3333445626 3952821602
 7175032540 5606572086 6237569496 6683426742 5035268417 3312463746
 9237487147 9713454201 1812226671 2460093296 4022692381 1267801529
 5009192468 4105942584 2355236719 9632980559 2212762406 8016466275
 1228935308 8199247926 4932578245 3992589015 2327054745 6889790653
 5270522918 3951163044 4520981383 6766324666 7551043473 7803770431
 3686273628 2215268322 7914558846 4423196276 1893548673 8836862149
 1913341282 6780386626 7098464262 7396836484 9553960148 6875724035
 7084181053 4186329201 3102490388 6313450246 9598377925 1278040507
 5261906348 6424972551 3533177779 7419370539 3083446565 8047497583
 5987464133 8780655195 8090604836 1202978606 5689129232 3014718150
 6829001048 5788646225 3841331821 7203076986 5316803628 6399991653
 5283227804 7707808405 8178619377 6502303487 2541442115 6097504486
 8850514709 5396497137 1435128522 3116656165 867527